In [ ]:
import sys
sys.path.insert(0, '..')

# Fuzzy Cognitive Map (FCM) Example: Public Support for Terrorism (PSOT)

This notebook ports the Public Support for Terrorism (PSOT) model, originally implemented in Mathematica, to the Python `fcm` library. The model is based on the work of [Alt, P. D. (2013)](https://citeseerx.ist.psu.edu/viewdoc/download?doi=10.1.1.695.882&rep=rep1&type=pdf).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from lib.fcm import FCM

## 1. Define the FCM Structure

In [ ]:
# Node mappings from psot-verts.csv
mapping = {
    1: "lead", 2: "pkg", 3: "rsrc", 4: "opp", 5: "pres", 6: "EFF", 7: "reli", 8: "socs", 9: "glry", 10: "ATT", 11: "duty", 12: "rwrd", 13: "MOTV", 14: "intl", 15: "rvng", 16: "cprop", 17: "desp", 18: "PLEG", 19: "intm", 20: "lvic", 21: "prsk", 22: "scst", 23: "ACR", 24: "id", 25: "shgr", 26: "ugb", 27: "env", 28: "impl", 29: "hsucc", 30: "mgtc", 31: "prop", 32: "efdoc", 33: "hfail", 50: "PSOT*"
}

# Edge weights based on psot-eds-upd.txt, assuming activationThreshold = 0.5
activation_threshold = 0.5
strongOrLink = activation_threshold / 2  # 0.25
orLink = strongOrLink - 0.05  # 0.20
weakOrLink = activation_threshold / 4  # 0.125
andLink = activation_threshold / 4  # 0.125

psot_edges = [
    (1, 6, strongOrLink), (2, 6, strongOrLink), (3, 6, strongOrLink), (4, 6, orLink), (5, 6, orLink), (30, 6, weakOrLink), (31, 6, weakOrLink), (32, 6, weakOrLink),
    (7, 10, strongOrLink), (8, 10, weakOrLink), (9, 10, orLink), (24, 10, strongOrLink),
    (10, 13, strongOrLink), (11, 13, strongOrLink), (12, 13, weakOrLink),
    (14, 18, strongOrLink), (15, 18, weakOrLink), (16, 18, orLink), (17, 18, orLink),
    (19, 23, strongOrLink), (20, 23, strongOrLink), (21, 23, -strongOrLink), (22, 23, -strongOrLink),
    (25, 11, orLink + 0.1),
    (24, 7, strongOrLink + 0.3), (24, 11, strongOrLink),
    (23, 19, strongOrLink), (23, 20, strongOrLink), (18, 14, strongOrLink),
    (6, 50, andLink), (13, 50, andLink), (18, 50, andLink), (23, 50, andLink),
    (50, 50, 0.1),
    (29, 13, orLink), (29, 21, -(orLink + 0.4)),
    (6, 29, weakOrLink),
    (33, 13, -orLink), (33, 21, (orLink + 0.4)),
    (26, 6, -strongOrLink), (26, 13, -strongOrLink),
    (6, 13, weakOrLink), (13, 18, weakOrLink), (18, 23, weakOrLink)
]

## 2. Create and Visualize the FCM

In [ ]:
psot_fcm = FCM("PSOT Model")
psot_fcm.add_weighted_edges_from(psot_edges)
nx.relabel_nodes(psot_fcm, mapping, copy=False)

plt.figure(figsize=(12, 12))
psot_fcm.draw()
plt.show()

## 3. Define and Run Simulation Scenarios

In [ ]:
def run_scenario(fcm, active_nodes, title):
    print(f"--- {title} ---")
    initial_vector = FCM.create_initial_vector(fcm, active_nodes)
    mask = np.zeros_like(initial_vector)
    history = fcm.evolve_to_limit(initial_vector, mask)
    print("Final state:")
    print(history[-1])
    return history

# Scenario 1
active_nodes_1 = ["lead", "pkg", "opp", "pres", "MOTV", "intl", "cprop", "desp", "PLEG", "intm", "lvic", "id", "mgtc"]
history1 = run_scenario(psot_fcm, active_nodes_1, "Scenario 1")

# Scenario 2
active_nodes_2 = ["lead", "pkg", "opp", "pres", "MOTV", "intl", "rvng", "desp", "intm", "lvic", "id", "mgtc", "hfail"]
history2 = run_scenario(psot_fcm, active_nodes_2, "Scenario 2")

# Scenario 3
active_nodes_3 = ["lead", "pkg", "pres", "MOTV", "intl", "rvng", "desp", "intm", "lvic", "id", "mgtc", "prop", "hfail"]
history3 = run_scenario(psot_fcm, active_nodes_3, "Scenario 3")